# Fine-tuning Parler-TTS

## Goal of this notebook

In the following notebook, we'll fine-tune [Parler-TTS Mini v0.1](https://huggingface.co/parler-tts/parler_tts_mini_v0.1) on a 5h subset of the [Jenny TTS dataset](https://github.com/dioco-group/jenny-tts-dataset), a 30 hours high-quality mono-speaker TTS dataset, from an Irish female speaker named Jenny.

In particular, we'll:
- Annotate the Jenny dataset with natural language speech description using [Data-Speech](https://github.com/huggingface/dataspeech).
- Fine-tune Parler-TTS with the created dataset.

**You should be able to adapt this notebook to your own datasets quite easily.**





## Prepare the Environment

Throughout this tutorial, we'll use a GPU. The runtime is already configured to use the free 16GB T4 GPU provided through Google Colab Free Tier, so all you need to do is hit "Connect T4" in the top right-hand corner of the screen.

##### <a name="installation"> We'll install Parler-TTS and Data-Speech from source in order to train our model.

In [ ]:
!git clone https://github.com/huggingface/dataspeech.git
!cd dataspeech
!pip install --quiet -r ./dataspeech/requirements.txt

Cloning into 'dataspeech'...
remote: Enumerating objects: 650, done.
remote: Counting objects: 100% (221/221), done.
remote: Compressing objects: 100% (90/90), done.
remote: Total 650 (delta 138), reused 131 (delta 131), pack-reused 429 (from 1)
Receiving objects: 100% (650/650), 158.78 KiB | 950.00 KiB/s, done.
Resolving deltas: 100% (395/395), done.
     - 49.0 MB 60.3 MB/s 0:00:03
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 15.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.2/125.2 kB 10.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.1/87.1 kB 7.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 5.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) .

In [ ]:
!git clone https://github.com/huggingface/parler-tts.git
%cd parler-tts
!pip install --quiet -e .[train]

Cloning into 'parler-tts'...
remote: Enumerating objects: 1084, done.
remote: Counting objects: 100% (446/446), done.
remote: Compressing objects: 100% (171/171), done.
remote: Total 1084 (delta 331), reused 275 (delta 275), pack-reused 638 (from 1)
Receiving objects: 100% (1084/1084), 349.12 KiB | 2.27 MiB/s, done.
Resolving deltas: 100% (691/691), done.
/content/parler-tts
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 5.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.2/64.2 kB 6.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━

On Colab, we need to run an additional set-up, that you can skip if you're on your local machine.

In [ ]:
!pip install --upgrade protobuf wandb==0.16.6

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 23.3 MB/s eta 0:00:00
  Attempting uninstall: wandb
    Found existing installation: wandb 0.18.7
    Uninstalling wandb-0.18.7:
      Successfully uninstalled wandb-0.18.7


You should link you Hugging Face account so that you can push model repositories on the Hub. This will allow you to save your trained models on the Hub so that you can share them with the community.

Run the command below and then enter an authentication token from https://huggingface.co/settings/tokens. Create a new token if you do not have one already. You should make sure that this token has "write" privileges.

In [ ]:
!git config --global credential.helper store
!huggingface-cli login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) Y
Token is valid (permission: write).
The token `Microsoft tts` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved in your configured git credential helpers (store).
Your token has been saved to /root/.cache

## 1. Creating our fine-tuning dataset


The aim here is to create an annotated version of Jenny TTS, in order to fine-tune the [Parler-TTS v0.1 checkpoint](https://huggingface.co/parler-tts/parler_tts_mini_v0.1) on this dataset.

Thanks to a [script similar to what's described in the Data-Speech FAQ](https://github.com/huggingface/dataspeech?tab=readme-ov-file#how-do-i-use-datasets-that-i-have-with-this-repository), we've uploaded the dataset to the HuggingFace hub, under the name [reach-vb/jenny_tts_dataset](https://huggingface.co/datasets/reach-vb/jenny_tts_dataset).

The purpose of this notebook is demonstration so we've pushed a 6h subset of the dataset that we'll work with: [ylacombe/jenny-tts-6h](https://huggingface.co/datasets/ylacombe/jenny-tts-6h).

Feel free to follow the link above to listen to some samples of the Jenny TTS dataset thanks to the hub viewer.

> Refer to the [Data-Speech README](https://github.com/huggingface/dataspeech?tab=readme-ov-file#data-speech) for more detailed explanations of what's going on under-the-hood.

We'll:
1. Annotate the Jenny dataset with continuous variables that measures the speech characteristics
2. Map those annotations to text bins that characterize the speech characteristics.
3. Create natural language descriptions from those text bins

In [ ]:
%cd ../dataspeech

/content/dataspeech


But first, let's look at a few samples from the Jenny dataset!

In [ ]:
from datasets import load_dataset
dataset = load_dataset("En1gma02/hindi_speech_male_5hr")

README.md:   0%|          | 0.00/476 [00:00<?, ?B/s]

train-00000-of-00009.parquet:   0%|          | 0.00/478M [00:00<?, ?B/s]

train-00001-of-00009.parquet:   0%|          | 0.00/461M [00:00<?, ?B/s]

train-00002-of-00009.parquet:   0%|          | 0.00/502M [00:00<?, ?B/s]

train-00003-of-00009.parquet:   0%|          | 0.00/291M [00:00<?, ?B/s]

...

train-00005-of-00009.parquet:   0%|          | 0.00/358M [00:00<?, ?B/s]

train-00006-of-00009.parquet:   0%|          | 0.00/602M [00:00<?, ?B/s]

train-00007-of-00009.parquet:   0%|          | 0.00/532M [00:00<?, ?B/s]

train-00008-of-00009.parquet:   0%|          | 0.00/514M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5842 [00:00<?, ? examples/s]

In [ ]:
from IPython.display import Audio
print(dataset["train"][0]["transcription"])
Audio(dataset["train"][0]["audio"]["array"], rate=dataset["train"][0]["audio"]["sampling_rate"])

prasidda kabIra adhyetA, puruShottama agravAla kA yaha shodha Alekha, usa rAmAnaMda kI khoja karatA hai


In [ ]:
from IPython.display import Audio
print(dataset["train"][1]["transcription"])
Audio(dataset["train"][1]["audio"]["array"], rate=dataset["train"][1]["audio"]["sampling_rate"])

kintu Adhunika pAMDitya, na sirfa eka brAhmaNa rAmAnaMda ke, eka julAhe kabIra kA guru hone se, balki donoM ke samakAlIna hone se bhI, inakAra karatA hai


In [ ]:
del dataset


### Annotating the Jenny dataset

We'll use [`main.py`](https://github.com/huggingface/dataspeech/blob/main/main.py) to get the following continuous variables:
- Speaking rate `(nb_phonemes / utterance_length)`
- Signal-to-noise ratio (SNR)
- Reverberation
- Speech monotony


In [ ]:
!python main.py "En1gma02/processed_hindi_speech_male_5hr" \
  --configuration "default" \
  --text_column_name "transcription" \
  --audio_column_name "audio" \
  --cpu_num_workers 2 \
  --num_workers_per_gpu_for_pitch 2 \
  --rename_column \
  --repo_id "En1gma02/hindi_speech_male_5hr_tagged"

2024-11-21 20:18:53.998027: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-11-21 20:18:54.018056: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-11-21 20:18:54.024167: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-11-21 20:18:54.038768: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-11-21 20:18:55.219128: W tensorflow/comp

The whole process took under 10mn!

The resulting dataset will be pushed to the HuggingFace hub under your HuggingFace handle. Mine was push to [ylacombe/jenny-tts-tags-6h](https://huggingface.co/datasets/ylacombe/jenny-tts-tags-6h).

Let's see what the new dataset looks like:

In [ ]:
from datasets import load_dataset
dataset = load_dataset("En1gma02/hindi_speech_male_5hr_tagged")
print("SNR 1st sample", dataset["train"][0]["snr"])
print("C50 2nd sample", dataset["train"][0]["c50"])
del dataset

README.md:   0%|          | 0.00/662 [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/1.47M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5842 [00:00<?, ? examples/s]

SNR 1st sample 65.81938934326172
C50 2nd sample 59.5479621887207


As you can see, the current annotations are continuous variables. To use it with Parler-TTS, we need to convert it to textual description, something that the two next steps will take care of.

### 2. Map annotations to text bins

Since the ultimate goal here is to fine-tune the [Parler-TTS v0.1 checkpoint](https://huggingface.co/parler-tts/parler_tts_mini_v0.1) on the Jenny dataset, we want to stay consistent with the text bins of the datasets on which the latter model was trained.

This is easy to do thanks to the following:

In [ ]:
!python ./scripts/metadata_to_text.py \
    "En1gma02/hindi_speech_male_5hr_tagged" \
    --repo_id "hindi_speech_male_5hr_tagged" \
    --configuration "default" \
    --cpu_num_workers 2 \
    --path_to_bin_edges "./examples/tags_to_annotations/v01_bin_edges.json" \
    --avoid_pitch_computation

Already computed bin edges have been passed for speaking_rate. Will use: [3.508771929824561, 6.187242299296628, 8.865712668768696, 11.544183038240764, 14.22265340771283, 16.901123777184896, 19.579594146656966, 22.258064516129032].
Map (num_proc=2): 100% 5842/5842 [00:05<00:00, 1061.47 examples/s]
Already computed bin edges have been passed for noise. Will use: [27.179607391357422, 33.90050179617746, 40.62139620099749, 47.342290605817524, 54.063185010637554, 60.78407941545759, 67.50497382027763, 74.22586822509766].
Map (num_proc=2): 100% 5842/5842 [00:03<00:00, 1616.92 examples/s]
Already computed bin edges have been passed for reverberation. Will use: [30.498437881469727, 34.706024169921875, 38.91361045837402, 43.12119674682617, 47.32878303527832, 51.53636932373047, 55.74395561218262, 59.951541900634766].
Map (num_proc=2): 100% 5842/5842 [00:03<00:00, 1604.69 examples/s]
Already computed bin edges have been passed for speech_monotony. Will use: [0.0, 17.430070059640066, 34.860140119280

Thanks to [`v01_bin_edges.json`](https://github.com/huggingface/dataspeech/blob/main/examples/tags_to_annotations/v01_bin_edges.json), we don't need to recompute bins from scratch and the above script takes a few seconds.

The resulting dataset will be pushed to the HuggingFace hub under your HuggingFace handle. Mine was push to [ylacombe/jenny-tts-tags-6h](https://huggingface.co/datasets/ylacombe/jenny-tts-tags-6h).

You can notice that text bins such as `quite noisy`, `very fast` have been added to the samples.

In [ ]:
from datasets import load_dataset
dataset = load_dataset("En1gma02/hindi_speech_male_5hr_tagged")
print("Noise 1st sample:", dataset["train"][0]["noise"])
print("Speaking rate 2nd sample:", dataset["train"][0]["speaking_rate"])
del dataset

README.md:   0%|          | 0.00/781 [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/1.43M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5842 [00:00<?, ? examples/s]

Noise 1st sample: quite clear
Speaking rate 2nd sample: very slowly



### 3. Create natural language descriptions from those text bins

Now that we have text bins associated to the Jenny dataset, the next step is to create natural language descriptions out of the few created features.

Here, we decided to create prompts that use the name `Jenny`, prompts that'll look like the following:
`In a very expressive voice, Jenny pronounces her words incredibly slowly. There's some background noise in this room with a bit of echo'`

This step generally demands more resources and times and should use one or many GPUs.

The following command shows how to do it using the [2B version of the Gemma model from Google](https://huggingface.co/google/gemma-2b-it), which should run in about 15 minutes in this Colab free T4.


As usual, we precise the dataset name and configuration we want to annotate. `model_name_or_path` should point to a `transformers` model for prompt annotation. You can find a list of such models [here](https://huggingface.co/models?pipeline_tag=text-generation&library=transformers&sort=trending).

**Note** how we've been able to specify that the dataset is mono-speaker and that we should name the voice Jenny thanks to the flags:


`--speaker_name "Jenny" --is_single_speaker`.


In [ ]:
%cd ..

/content


In [ ]:
!python ./dataspeech/scripts/run_prompt_creation.py \
  --speaker_name "Male Hindi Speaker" \
  --is_single_speaker \
  --dataset_name "En1gma02/hindi_speech_male_5hr_tagged" \
  --output_dir "./tmp_hindi" \
  --dataset_config_name "default" \
  --model_name_or_path "google/gemma-2b-it" \
  --per_device_eval_batch_size 12 \
  --attn_implementation "sdpa" \
  --dataloader_num_workers 2 \
  --push_to_hub \
  --hub_dataset_id "hindi_speech_male_5hr_tagged" \
  --preprocessing_num_workers 2

11/21/2024 20:33:02 - INFO - __main__ - *** Load annotated dataset ***
11/21/2024 20:33:04 - INFO - __main__ - *** Load pretrained model ***
config.json: 100% 627/627 [00:00<00:00, 5.17MB/s]
2024-11-21 20:33:06.785502: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-11-21 20:33:06.805390: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-11-21 20:33:06.811393: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-11-21 20:33:06.826005: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical

Let's take a look at some created prompts:

In [ ]:
from datasets import load_dataset
dataset = load_dataset("En1gma02/indian_accent_english_tagged")
print("1st sample:", dataset["train"][0]["text_description"])
print("2nd sample:", dataset["train"][1]["text_description"])
del dataset

KeyError: 'text_description'

**Observation:** The first sample unfortunately doesn't have the name Jenny in it. This is probably because we use a smaller and thus less precise model that one we would have gone for if this notebook had more resources (e.g we've used [Mistral 7B v2](https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.2) to create the Parler-TTS training dataset). This shouldn't prevent our model to learn what we want though.